In [210]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

In [211]:
df = pd.read_csv('../data/bank-additional-full.csv', sep=';')

## Removes Nulls & Drops Irrelevent Columns

In [212]:
# Where housing is missing, loan is missing
# Remove those rows
df = df[(df['housing'] != 'unknown') & (df['loan'] != 'unknown')]
# Drop jobs unknowns
df = df[df['job'] != 'unknown']
df = df[df['marital'] != 'unknown']
# Drop jobs unknowns
df = df[df['education'] != 'unknown']
cols_to_drop = ['default', 'duration', 'previous', 'pdays' ]
df.drop(columns=cols_to_drop, inplace=True)

Make plot to describe why scaling certain features (campaign)

In [213]:
# Encode target
y = (df['y'] == 'yes').astype(int)
X = df.drop(columns=['y'])

In [214]:
categorical_cols = ['job', 'marital', 'education', 'housing', 'loan',
                    'contact', 'month', 'day_of_week', 'poutcome']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=int)

## LogReg, use scaling

Keep campaign, scale campaign

In [215]:
X_train_log, X_test_log, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_log[numeric_cols] = scaler.fit_transform(X_train_log[numeric_cols])
X_test_log[numeric_cols] = scaler.transform(X_test_log[numeric_cols])

print(X_train_log.shape, X_test_log.shape, y_train.mean().round(3))

(30596, 43) (7649, 43) 0.111


## Random Forest, don't use scaling

In [216]:
# Evaluation for Random Forest
def evaluate_rf(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred, digits=3))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba):.3f}")
    print(f"PR-AUC:   {average_precision_score(y_test, y_proba):.3f}")

In [217]:
# Train/Test/Split Data
X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model 1

In [218]:
# Train Model
rf1 = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
rf1.fit(X_train_rf, y_train)

# Results
evaluate_rf(rf1, X_test_rf, y_test)

              precision    recall  f1-score   support

           0      0.937     0.926     0.931      6797
           1      0.459     0.500     0.479       852

    accuracy                          0.879      7649
   macro avg      0.698     0.713     0.705      7649
weighted avg      0.883     0.879     0.881      7649

Confusion matrix:
 [[6295  502]
 [ 426  426]]
ROC-AUC:  0.787
PR-AUC:   0.456


# Model 2

In [219]:
# Train Model
rf2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf2.fit(X_train_rf, y_train)

# Results
evaluate_rf(rf2, X_test_rf, y_test)

              precision    recall  f1-score   support

           0      0.947     0.882     0.913      6797
           1      0.392     0.607     0.476       852

    accuracy                          0.851      7649
   macro avg      0.670     0.744     0.695      7649
weighted avg      0.885     0.851     0.865      7649

Confusion matrix:
 [[5996  801]
 [ 335  517]]
ROC-AUC:  0.801
PR-AUC:   0.471


In [220]:
(df == 'unknown').sum()

age               0
job               0
marital           0
education         0
housing           0
loan              0
contact           0
month             0
day_of_week       0
campaign          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64